Simulator: No Lens Light
========================

This script simulates `Imaging` of a 'galaxy-scale' which is identical to the `simple` simulated in the `start_here.py`
script, but where the lens galaxy's light is omitted.

It is used in `autolens_workspace/notebooks/imaging/features/no_lens_light/modeling.ipynb` to illustrate how to fit a
lens model to data where the lens galaxy's light is not present (e.g. because it is too faint to be detected).

__Contents__

- **Model:** Compose the lens model fitted to the data.
- **Dataset Paths:** The `dataset_type` describes the type of data being simulated and `dataset_name` gives it a.
- **Simulate:** Simulate the image using a (y,x) grid with the adaptive over sampling scheme.
- **Ray Tracing:** Setup the lens galaxy's light, mass and source galaxy light for this simulated lens.
- **Output:** Output the simulated dataset to the dataset path as .fits files.
- **Visualize:** Output a subplot of the simulated dataset, the image and the tracer's quantities to the dataset.
- **Mask Extra Galaxies:** Save a `mask_extra_galaxies.fits` covering the extra galaxy for noise-scaling tutorials.
- **Tracer json:** Save the `Tracer` in the dataset folder as a .json file, ensuring the true light profiles, mass.

__Model__

This script simulates `Imaging` of a 'galaxy-scale' strong lens where:

 - The lens galaxy's total mass distribution is an `Isothermal` and `ExternalShear`.
 - The source galaxy's light is an `Sersic`.
 - A faint extra galaxy is included offset from the lens, whose emission must be removed via noise scaling
   (a `mask_extra_galaxies.fits` covering it is written below).

__Start Here Notebook__

If any code in this script is unclear, refer to the `imaging/simulator.ipynb` notebook.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

# from autolens import setup_notebook; setup_notebook()

from pathlib import Path
import autolens as al
import autolens.plot as aplt

__Dataset Paths__

The `dataset_type` describes the type of data being simulated and `dataset_name` gives it a descriptive name. 

In [ ]:
dataset_type = "imaging"
dataset_name = "simple__no_lens_light"
dataset_path = Path("dataset", dataset_type, dataset_name)

__Simulate__

Simulate the image using a (y,x) grid with the adaptive over sampling scheme.

In [ ]:
grid = al.Grid2D.uniform(
    shape_native=(100, 100),
    pixel_scales=0.1,
)

The centre of a faint extra galaxy, placed inside the 3.0" modeling mask but clear of the lensed source arcs
(Einstein radius ~1.6"). It is reused for over-sampling, the galaxy itself and the `mask_extra_galaxies.fits`.

In [ ]:
extra_galaxy_centre = (2.2, 1.6)

over_sample_size = al.util.over_sample.over_sample_size_via_radial_bins_from(
    grid=grid,
    sub_size_list=[32, 8, 2],
    radial_list=[0.3, 0.6],
    centre_list=[(0.0, 0.0), extra_galaxy_centre],
)

grid = grid.apply_over_sampling(over_sample_size=over_sample_size)

Simulate a simple Gaussian PSF for the image.

In [ ]:
psf = al.Convolver.from_gaussian(
    convolve_over_sample_size=1,
    shape_native=(11, 11),
    sigma=0.1,
    pixel_scales=grid.pixel_scales,
)

Create the simulator for the imaging data, which defines the exposure time, background sky, noise levels and psf.

In [ ]:
simulator = al.SimulatorImaging(
    exposure_time=300.0,
    psf=psf,
    background_sky_level=0.1,
    add_poisson_noise_to_data=True,
)

__Ray Tracing__

Setup the lens galaxy's light, mass and source galaxy light for this simulated lens.

the `lens_galaxy` below does not include a `bulge` or `disk` component and therefore has no lens light.

In [ ]:
lens_galaxy = al.Galaxy(
    redshift=0.5,
    mass=al.mp.Isothermal(
        centre=(0.0, 0.0),
        einstein_radius=1.6,
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
    ),
    shear=al.mp.ExternalShear(gamma_1=0.05, gamma_2=0.05),
)

source_galaxy = al.Galaxy(
    redshift=1.0,
    bulge=al.lp.SersicCore(
        centre=(0.0, 0.0),
        ell_comps=al.convert.ell_comps_from(axis_ratio=0.8, angle=60.0),
        intensity=4.0,
        effective_radius=0.1,
        sersic_index=1.0,
    ),
)

A single faint extra galaxy offset from the lens, representing a nearby contaminating object. It has a light
profile only (no mass), so the lensed source arcs are unchanged; its emission is removed in the fit examples via
the `__Extra Galaxies Noise Scaling__` step using the `mask_extra_galaxies.fits` written below.

In [ ]:
extra_galaxy = al.Galaxy(
    redshift=0.5,
    light=al.lp.ExponentialSph(
        centre=extra_galaxy_centre, intensity=1.0, effective_radius=0.3
    ),
)

Use these galaxies to setup a tracer, which will generate the image for the simulated `Imaging` dataset.

In [ ]:
tracer = al.Tracer(galaxies=[lens_galaxy, extra_galaxy, source_galaxy])

Lets look at the tracer`s image, this is the image we'll be simulating.

In [ ]:
aplt.plot_array(array=tracer.image_2d_from(grid=grid), title="Image")

Pass the simulator a tracer, which creates the image which is simulated as an imaging dataset.

In [ ]:
dataset = simulator.via_tracer_from(tracer=tracer, grid=grid)

Plot the simulated `Imaging` dataset before outputting it to fits.

In [ ]:
aplt.subplot_imaging_dataset(dataset=dataset)

__Output__

Output the simulated dataset to the dataset path as .fits files.

In [ ]:
aplt.fits_imaging(
    dataset=dataset,
    data_path=dataset_path / "data.fits",
    psf_path=dataset_path / "psf.fits",
    noise_map_path=dataset_path / "noise_map.fits",
    overwrite=True,
)

__Visualize__

Output a subplot of the simulated dataset, the image and the tracer's quantities to the dataset path as .png files.

In [ ]:

aplt.subplot_imaging_dataset(dataset=dataset)
aplt.plot_array(array=dataset.data, title="Data")

aplt.subplot_tracer(
    tracer=tracer, grid=grid, output_path=dataset_path, output_format="png"
)
aplt.subplot_galaxies_images(
    tracer=tracer, grid=grid, output_path=dataset_path, output_format="png"
)

__Mask Extra Galaxies__

Build and output a `mask_extra_galaxies.fits` covering the extra galaxy, so the fit example (`imaging/fit.py`)
and the pixelization tutorials that load this dataset (`imaging/features/pixelization/modeling.py`,
`imaging/features/pixelization/fit.py`) can demonstrate the noise-scaling API on a real contaminant.

The circle is sized to ~3x the galaxy's `effective_radius`, derived from the same `extra_galaxy_centre` defined
above so it stays in sync. The mask shape tracks `dataset.shape_native`, so `PYAUTO_SMALL_DATASETS=1` is honoured
automatically.

In [ ]:
mask_extra_galaxies = al.Mask2D.circular(
    shape_native=dataset.shape_native,
    pixel_scales=dataset.pixel_scales,
    centre=extra_galaxy_centre,
    radius=3.0 * 0.3,
    invert=True,  # `True` inside the circle, i.e. the region whose noise is scaled.
)

aplt.fits_array(
    array=mask_extra_galaxies,
    file_path=dataset_path / "mask_extra_galaxies.fits",
    overwrite=True,
)

__Tracer json__

Save the `Tracer` in the dataset folder as a .json file, ensuring the true light profiles, mass profiles and galaxies
are safely stored and available to check how the dataset was simulated in the future.

This can be loaded via the method `tracer = al.from_json()`.

In [ ]:
al.output_to_json(
    obj=tracer,
    file_path=Path(dataset_path, "tracer.json"),
)

__Multiple Images__

Lens modeling can use a "positions likelihood penalty", whereby mass models which traces the (y,x) 
coordinates of multiple images of a source galaxy to positions which are far apart from one another 
in the source plane are penalized in the lens model's overall likelihood.

This speeds up lens modeling, helps the non-linear search avoid local maxima and is vital for inferred 
accurate solutions when using pixelized source reconstructions.

For real data, the multiple image positions are determined by eye from the data, for example
using a Graphical User Interface (GUI) to mark them with mouse clicks. For simulated data, we can save
ourselves time by using the `PointSolver` to determine the multiple image positions automatically and
output to a .json file.

If you have not looked in the `point_source` package, the point solver is the core tool used to find
multiple image positions for point source lens modeling (e.g. lensed quasars).

In [ ]:
solver = al.PointSolver.for_grid(
    grid=grid, pixel_scale_precision=0.001, magnification_threshold=0.1
)

positions = solver.solve(
    tracer=tracer, source_plane_coordinate=source_galaxy.bulge.centre
)

al.output_to_json(
    file_path=dataset_path / "positions.json",
    obj=positions,
)

The dataset can be viewed in the folder `autolens_workspace/dataset/imaging/simple__no_lens_light`.